# Trail Digital Twin Extensions

This notebook extends the Sensors reproduction with repository-specific models: Stage 4 segment regression, HR-informed submaximal transfer, REDI load variants, in-activity acute TRIMP, VMA 18 km/h anchoring, and an extension-only segment-level grid search.

The segment grid search is not a paper replication. Its objective fits `actualTimeSec` at segment grain and then reports race-summed metrics as a guardrail.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from services import trail_performance_model as tpm

DATA_DIR = PROJECT_ROOT / "data"
TIMESERIES_DIR = DATA_DIR / "timeseries"
METRICS_TS_DIR = DATA_DIR / "metrics_ts"
RAW_STRAVA_DIR = DATA_DIR / "raw" / "strava"
SEGMENT_KM = 1.0

SELECTED_RACE_DATE_STRINGS = [
    "2025-10-18",
    "2026-04-12",
    "2025-08-06",
    "2025-09-28",
    "2025-06-23",
    "2025-07-27",
    "2025-07-24",
    "2025-08-01",
    "2024-08-18",
    "2025-11-01",
    "2025-04-27",
    "2025-07-12",
    "2026-02-22",
    "2026-04-18",
    "2024-07-28",
    "2025-08-03",
    "2025-06-29",
    "2025-08-23",
]

CTL_WEIGHT = 0.05
TSB_WEIGHT = 0.10
CTL_FACTOR_MIN = 0.90
CTL_FACTOR_MAX = 1.08

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (9, 5)

VMA_FLAT_KMH = 18.0


## Load Data and Physiology

The extension keeps the same local data extraction as the reproduction notebook, but uses `VMA_FLAT_KMH = 18.0` as the flat anchor for extension-only grid models.

In [ ]:
activities = pd.read_csv(DATA_DIR / "activities.csv", dtype={"activityId": str})
activity_metrics = pd.read_csv(DATA_DIR / "activities_metrics.csv", dtype={"activityId": str})
athletes = pd.read_csv(DATA_DIR / "athlete.csv")
thresholds = pd.read_csv(DATA_DIR / "thresholds.csv")
daily_metrics = pd.read_csv(DATA_DIR / "daily_metrics.csv")

metric_cols = [
    "activityId",
    "category",
    "distanceEqKm",
    "trimp",
    "hrSpeedShift",
    "hrZone_z1_upper",
    "hrZone_z2_upper",
    "hrZone_z3_upper",
    "hrZone_z4_upper",
]
metric_cols = [col for col in metric_cols if col in activity_metrics.columns]
activity_df = activities.merge(activity_metrics[metric_cols], on="activityId", how="left")

athlete = athletes.iloc[0]
HR_REST = float(athlete.get("hrRest", 60.0))
HR_MAX = float(athlete.get("hrMax", 190.0))
threshold_30 = thresholds[thresholds["name"].astype(str).eq("Threshold 30")]
if not threshold_30.empty:
    threshold_row = threshold_30.iloc[0]
    V_VT2_KMH = float(
        np.nanmean([
            pd.to_numeric(threshold_row.get("paceFlatKmhMin"), errors="coerce"),
            pd.to_numeric(threshold_row.get("paceFlatKmhMax"), errors="coerce"),
        ])
    )
else:
    V_VT2_KMH = 15.0

activity_df = tpm.add_hr_reserve(activity_df, hr_rest=HR_REST, hr_max=HR_MAX)
activity_df["actualTimeSec"] = pd.to_numeric(activity_df["movingSec"], errors="coerce")
activity_df["actualTimeSec"] = activity_df["actualTimeSec"].where(
    activity_df["actualTimeSec"] > 0,
    pd.to_numeric(activity_df["elapsedSec"], errors="coerce"),
)
activity_df["startDate"] = pd.to_datetime(activity_df["startTime"], errors="coerce").dt.date
activity_df["category"] = activity_df["category"].astype(str)

print(f"Activities loaded: {len(activity_df)}")
print(f"HR rest/max: {HR_REST:.0f}/{HR_MAX:.0f} bpm")
print(f"VT2 proxy speed from Threshold 30: {V_VT2_KMH:.2f} km/h")
display(activity_df["category"].fillna("missing").value_counts().rename("count").to_frame())


## Load Features

Training load variants are computed from all activities: TRIMP CTL/ATL/TSB, REDI, distance-equivalent CTL/TSB, and vertical-load CTL/TSB.

In [ ]:
def rename_load_features(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return df.rename(
        columns={
            "load": f"{prefix}Load",
            "ctl": f"{prefix}Ctl",
            "atl": f"{prefix}Atl",
            "tsb": f"{prefix}Tsb",
        }
    )

trimp_load = rename_load_features(tpm.compute_ctl_atl_tsb(daily_metrics, load_col="trimp"), "trimp")
distance_eq_load = rename_load_features(
    tpm.compute_ctl_atl_tsb(daily_metrics, load_col="distanceEqKm"),
    "distanceEq",
)
vertical_load = rename_load_features(tpm.compute_ctl_atl_tsb(daily_metrics, load_col="ascentM"), "vertical")
redi_load = tpm.compute_redi_load_features(daily_metrics, load_col="trimp", prefix="trimp")

for feature_df in [trimp_load, distance_eq_load, vertical_load, redi_load]:
    cols = [col for col in feature_df.columns if col != "date"]
    activity_df = tpm.attach_previous_daily_features(activity_df, feature_df, feature_cols=cols)

activity_df = activity_df.rename(columns={"trimpCtl": "ctl", "trimpAtl": "atl", "trimpTsb": "tsb"})

weather_records = []
for activity_id in activity_df["activityId"].astype(str):
    raw_path = RAW_STRAVA_DIR / f"{activity_id}.json"
    parsed = {"activityId": activity_id, "temperatureC": np.nan, "weatherSource": ""}
    if raw_path.exists():
        try:
            detail = json.loads(raw_path.read_text())
            temperature = pd.to_numeric(detail.get("average_temp"), errors="coerce")
            if pd.notna(temperature):
                parsed["temperatureC"] = float(temperature)
                parsed["weatherSource"] = "strava_average_temp"
        except Exception:
            pass
    weather_records.append(parsed)
weather_df = pd.DataFrame(weather_records)
activity_df = activity_df.merge(weather_df, on="activityId", how="left")

print(f"Training-load daily rows: {len(trimp_load)}")
print(f"Activities with Strava average_temp: {weather_df['temperatureC'].notna().sum()}")


## Segment Extraction and Cohorts

The extension evaluates the same hard, top-10 HRR, and selected-date cohorts used by the paper reproduction.

In [ ]:
def load_processed_timeseries(activity_id: str) -> pd.DataFrame:
    raw_path = TIMESERIES_DIR / f"{activity_id}.csv"
    if raw_path.exists():
        raw_df = pd.read_csv(raw_path)
        processed = tpm.prepare_raw_timeseries_for_segments(raw_df)
        if not processed.empty and "cumulated_distance" in processed.columns:
            if "speed_km_h" in processed.columns and "grade_ma_10" in processed.columns:
                processed["speed_eq_km_h"] = processed["speed_km_h"] * processed["grade_ma_10"].map(tpm.gap_factor)
            return processed

    metrics_path = METRICS_TS_DIR / f"{activity_id}.csv"
    if metrics_path.exists():
        return pd.read_csv(metrics_path)
    return pd.DataFrame()

segment_candidates = activity_df[
    activity_df["category"].astype(str).str.upper().isin(["TRAIL_RUN", "RUN"])
    & activity_df["hasTimeseries"].astype(str).str.lower().isin(["true", "1", "yes"])
].copy()

segments_by_activity: dict[str, pd.DataFrame] = {}
segment_frames = []
qc_rows = []
for _, row in segment_candidates.iterrows():
    activity_id = str(row["activityId"])
    ts_df = load_processed_timeseries(activity_id)
    segments = tpm.segment_timeseries(ts_df, segment_km=SEGMENT_KM, hr_rest=HR_REST, hr_max=HR_MAX)
    usable = not segments.empty and segments["distanceKm"].sum() > 0 and segments["actualTimeSec"].sum() > 0
    qc_rows.append(
        {
            "activityId": activity_id,
            "name": row.get("name"),
            "category": row.get("category"),
            "usableSegments": usable,
            "segmentCount": len(segments),
            "segmentDistanceKm": float(segments["distanceKm"].sum()) if usable else 0.0,
            "segmentTimeSec": float(segments["actualTimeSec"].sum()) if usable else 0.0,
        }
    )
    if usable:
        segments = segments.copy()
        segments["activityId"] = activity_id
        if "terrainFamily" not in segments.columns:
            segments["terrainFamily"] = segments["avgGrade"].map(tpm.terrain_family)
        else:
            segments["terrainFamily"] = segments["terrainFamily"].fillna(segments["avgGrade"].map(tpm.terrain_family))
        segments_by_activity[activity_id] = segments
        segment_frames.append(segments)

qc_df = pd.DataFrame(qc_rows)
all_segments_df = pd.concat(segment_frames, ignore_index=True) if segment_frames else pd.DataFrame()
segment_summary = (
    all_segments_df.groupby("activityId")
    .agg(
        technicalityGps=("technicalityGps", "mean"),
        meanAltitudeM=("meanAltitudeM", "mean"),
        segmentDistanceKm=("distanceKm", "sum"),
        segmentActualTimeSec=("actualTimeSec", "sum"),
        meanSegmentHrReserve=("meanHrReserve", "mean"),
        meanSpeedEqKmh=("meanSpeedEqKmh", "mean"),
    )
    .reset_index()
    if not all_segments_df.empty
    else pd.DataFrame(columns=["activityId"])
)
activity_df = activity_df.merge(segment_summary, on="activityId", how="left")
activity_df["usableSegmentActivity"] = activity_df["activityId"].astype(str).isin(segments_by_activity)
activity_df["usableTrailRun"] = activity_df["category"].astype(str).str.upper().eq("TRAIL_RUN") & activity_df["usableSegmentActivity"]
activity_df["hardTrailRun"] = tpm.hard_trailrun_mask(activity_df) & activity_df["usableSegmentActivity"]

print(f"Usable run/trail activities with segments: {len(segments_by_activity)}")
print(f"Usable TrailRuns: {int(activity_df['usableTrailRun'].sum())}")
print(f"Hard race-like TrailRuns: {int(activity_df['hardTrailRun'].sum())}")
display(qc_df.head(10))


In [ ]:
def add_readiness_factors(df: pd.DataFrame, value_col: str, balance_col: str, output_col: str) -> pd.DataFrame:
    result = df.copy()
    reference = float(pd.to_numeric(result[value_col], errors="coerce").median())
    if not np.isfinite(reference) or reference <= 0:
        reference = 1.0
    result[output_col] = result.apply(
        lambda row: tpm.ctl_readiness_factor(
            row.get(value_col, np.nan),
            row.get(balance_col, 0.0),
            ctl_reference=reference,
            ctl_weight=CTL_WEIGHT,
            tsb_weight=TSB_WEIGHT,
            min_factor=CTL_FACTOR_MIN,
            max_factor=CTL_FACTOR_MAX,
        ),
        axis=1,
    )
    return result

activity_df = add_readiness_factors(activity_df, "ctl", "tsb", "ctlReadinessFactor")
activity_df = add_readiness_factors(
    activity_df,
    "trimpRediSlow",
    "trimpRediBalance",
    "rediReadinessFactor",
)

hard_activity_df = activity_df[activity_df["hardTrailRun"]].copy()
top10_ids = tpm.top_hrr_hard_trailrun_ids(activity_df, n=10)
top10_activity_df = activity_df[activity_df["activityId"].astype(str).isin(top10_ids)].copy()
selected_activity_df = tpm.select_best_activity_by_dates(activity_df, SELECTED_RACE_DATE_STRINGS)
selected_activity_df = selected_activity_df[
    selected_activity_df["activityId"].astype(str).isin(segments_by_activity)
].copy()

cohorts = {
    "hardTrailRun": hard_activity_df,
    "top10HardTrailByHRR": top10_activity_df,
    "selectedDateRaces": selected_activity_df,
}
cohort_counts = pd.DataFrame(
    [
        {
            "cohort": name,
            "activities": len(df),
            "distanceKm": pd.to_numeric(df.get("distanceKm"), errors="coerce").sum(),
            "ascentM": pd.to_numeric(df.get("ascentM"), errors="coerce").sum(),
        }
        for name, df in cohorts.items()
    ]
)
display(cohort_counts)
display(selected_activity_df[["requestedDate", "activityId", "name", "category", "distanceKm", "ascentM", "hrReserveRatio"]].sort_values("requestedDate"))


## Progressive Linear Regression Diagnostic

This section keeps the all-linear log-time regression only as a diagnostic for variable signs and relative importance. It is not the main extension model, because the paper-style extension below keeps the original speed-equation structure.

The diagnostic equation is:

$$\log(t_{rs})=\beta_0+\beta_d\log(d_{rs})+\beta_g\log(f_{GAP,rs})+\beta_a(1-f_{alt,rs})+\beta_p p_{rs}+\beta_h HRR_{rs}+\beta_l L_r+\beta_u U_{rs}+\epsilon_{rs}$$

Here $p_{rs}$ is the original progress-based linear fatigue proxy. After E6, progress is removed because acute TRIMP is intended to carry the in-activity fatigue state. E6 uses the exponentially decayed acute TRIMP state $D_{rs}$ only; E6bis uses the cumulative acute TRIMP state $C_{rs}$ only.

In [ ]:
VMA_FLAT_KMH = 18.0


def metrics_row(cohort: str, stage: str, actual, predicted) -> dict[str, object]:
    metrics = tpm.regression_metrics(actual, predicted)
    return {
        "cohort": cohort,
        "stage": stage,
        "r2": metrics["r2"],
        "maeMin": metrics["maeSec"] / 60.0,
        "mapePct": metrics["mapePct"],
        "biasMin": metrics["biasSec"] / 60.0,
    }


def add_segment_model_features(segments: pd.DataFrame, activity_features: pd.DataFrame) -> pd.DataFrame:
    df = segments.merge(activity_features, on="activityId", how="left", suffixes=("", "_activity"))
    df["logActualTimeSec"] = np.log(pd.to_numeric(df["actualTimeSec"], errors="coerce").clip(lower=1.0))
    df["logDistanceKm"] = np.log(pd.to_numeric(df["distanceKm"], errors="coerce").clip(lower=1e-3))
    fallback_gap = pd.to_numeric(df["avgGrade"], errors="coerce").fillna(0.0).map(tpm.gap_factor)
    integrated_gap = df.get("gapFactorIntegrated", pd.Series(np.nan, index=df.index))
    df["gapFactor"] = pd.to_numeric(integrated_gap, errors="coerce").fillna(fallback_gap)
    df["logGapFactor"] = np.log(df["gapFactor"].clip(lower=1e-6))
    df["altitudePenalty"] = 1.0 - pd.to_numeric(df["meanAltitudeM"], errors="coerce").fillna(0.0).map(tpm.altitude_factor)
    df["ascentPerKm"] = pd.to_numeric(df["elevGainM"], errors="coerce").fillna(0.0) / df["distanceKm"].clip(lower=1e-3)
    df["descentPerKm"] = pd.to_numeric(df["elevLossM"], errors="coerce").fillna(0.0) / df["distanceKm"].clip(lower=1e-3)
    df["meanHrReserve"] = pd.to_numeric(df["meanHrReserve"], errors="coerce")
    df["technicalityCombined"] = pd.to_numeric(df.get("technicalityGps"), errors="coerce").fillna(0.0)
    df["temperatureC"] = pd.to_numeric(df.get("temperatureC", pd.Series(np.nan, index=df.index)), errors="coerce")
    for col in [
        "ctl",
        "tsb",
        "trimpRediSlow",
        "trimpRediBalance",
        "distanceEqCtl",
        "distanceEqTsb",
        "verticalCtl",
        "verticalTsb",
        "cumTrimpBefore",
        "decayedTrimpBefore",
    ]:
        df[col] = pd.to_numeric(df.get(col, pd.Series(np.nan, index=df.index)), errors="coerce")
    dummies = pd.get_dummies(df["terrainFamily"], prefix="terrain", dtype=float)
    return pd.concat([df, dummies], axis=1)


def linear_coefficient_frame(
    cohort_name: str,
    stage_name: str,
    df: pd.DataFrame,
    fitted: tpm.RegressionModel,
    target_col: str = "logActualTimeSec",
) -> pd.DataFrame:
    target = pd.to_numeric(df[target_col], errors="coerce")
    target_std = float(target.std(ddof=0)) if target.notna().any() else np.nan
    rows = [
        {
            "cohort": cohort_name,
            "stage": stage_name,
            "feature": "intercept",
            "coefficient": float(fitted.coefficients[0]),
            "standardizedCoefficient": np.nan,
            "absStandardizedCoefficient": np.nan,
        }
    ]
    for feature, coefficient in zip(fitted.feature_cols, fitted.coefficients[1:]):
        values = pd.to_numeric(df[feature], errors="coerce")
        feature_std = float(values.std(ddof=0)) if values.notna().any() else np.nan
        standardized = coefficient * feature_std / target_std if target_std and np.isfinite(target_std) else np.nan
        rows.append(
            {
                "cohort": cohort_name,
                "stage": stage_name,
                "feature": feature,
                "coefficient": float(coefficient),
                "standardizedCoefficient": float(standardized) if np.isfinite(standardized) else np.nan,
                "absStandardizedCoefficient": abs(float(standardized)) if np.isfinite(standardized) else np.nan,
            }
        )
    return pd.DataFrame(rows)


all_segments_with_trimp = tpm.add_in_activity_trimp_features(all_segments_df, decay_lambda=0.30)
segment_features = add_segment_model_features(all_segments_with_trimp, activity_df)
terrain_cols = [col for col in segment_features.columns if col.startswith("terrain_")]

progressive_feature_sets = {
    "E1 distance": ["logDistanceKm"],
    "E2 terrain physics + progress": ["logDistanceKm", "logGapFactor", "altitudePenalty", "progress", *terrain_cols],
    "E3 CTL readiness": ["logDistanceKm", "logGapFactor", "altitudePenalty", "progress", "ctl", "tsb", *terrain_cols],
    "E4 REDI readiness": ["logDistanceKm", "logGapFactor", "altitudePenalty", "progress", "trimpRediSlow", "trimpRediBalance", *terrain_cols],
    "E5 HR effort": ["logDistanceKm", "logGapFactor", "altitudePenalty", "progress", "meanHrReserve", "ctl", "tsb", *terrain_cols],
    "E6 decayed acute TRIMP": ["logDistanceKm", "logGapFactor", "altitudePenalty", "meanHrReserve", "ctl", "tsb", "decayedTrimpBefore", *terrain_cols],
    "E6bis cumulative acute TRIMP": ["logDistanceKm", "logGapFactor", "altitudePenalty", "meanHrReserve", "ctl", "tsb", "cumTrimpBefore", *terrain_cols],
}
LINEAR_LOO_STAGES = ["E6 decayed acute TRIMP", "E6bis cumulative acute TRIMP"]
LINEAR_FINAL_STAGE = "E6 decayed acute TRIMP"

extension_rows = []
linear_loo_frames = []
linear_coefficient_frames = []
for cohort_name, cohort_df in cohorts.items():
    ids = cohort_df["activityId"].astype(str).tolist()
    cohort_segments = segment_features[segment_features["activityId"].isin(ids)].copy()
    if cohort_segments.empty:
        continue
    actual_activity = cohort_df.set_index("activityId").loc[ids, "actualTimeSec"].astype(float)
    for stage_name, features in progressive_feature_sets.items():
        fitted = tpm.fit_linear_regression(cohort_segments, features, "logActualTimeSec")
        predicted_segments = np.exp(tpm.predict_linear_regression(cohort_segments, fitted))
        predicted_activity = (
            pd.DataFrame({"activityId": cohort_segments["activityId"], "pred": predicted_segments})
            .groupby("activityId")["pred"]
            .sum()
            .reindex(ids)
        )
        extension_rows.append(metrics_row(cohort_name, stage_name, actual_activity, predicted_activity))
        linear_coefficient_frames.append(
            linear_coefficient_frame(cohort_name, stage_name, cohort_segments, fitted)
        )

    for stage_name in LINEAR_LOO_STAGES:
        features = progressive_feature_sets[stage_name]
        loo_rows = []
        for held_out in ids:
            train = cohort_segments[cohort_segments["activityId"].ne(held_out)]
            test = cohort_segments[cohort_segments["activityId"].eq(held_out)]
            if train.empty or test.empty:
                continue
            fitted = tpm.fit_linear_regression(train, features, "logActualTimeSec")
            predicted = float(np.exp(tpm.predict_linear_regression(test, fitted)).sum())
            actual = float(actual_activity.loc[held_out])
            loo_rows.append(
                {
                    "cohort": cohort_name,
                    "stage": f"{stage_name} LOO",
                    "activityId": held_out,
                    "actualTimeSec": actual,
                    "predictedTimeSec": predicted,
                }
            )
        loo_df = pd.DataFrame(loo_rows)
        if not loo_df.empty:
            linear_loo_frames.append(loo_df)
            extension_rows.append(
                metrics_row(
                    cohort_name,
                    f"{stage_name} LOO",
                    loo_df["actualTimeSec"],
                    loo_df["predictedTimeSec"],
                )
            )

extension_comparison = pd.DataFrame(extension_rows)
linear_loo = pd.concat(linear_loo_frames, ignore_index=True) if linear_loo_frames else pd.DataFrame()
linear_variable_importance = (
    pd.concat(linear_coefficient_frames, ignore_index=True)
    if linear_coefficient_frames
    else pd.DataFrame()
)

final_linear_importance = linear_variable_importance[
    linear_variable_importance["stage"].isin(LINEAR_LOO_STAGES)
].sort_values(["cohort", "stage", "absStandardizedCoefficient"], ascending=[True, True, False])

display(extension_comparison.sort_values(["cohort", "maeMin"]))
display(final_linear_importance)


## HR-Informed Linear Models

These remain log-time OLS baselines trained with leave-one-activity-out validation on all usable TrailRuns. The segment model now uses the E6 decayed-TRIMP feature set, not the removed full E7 specification.

In [ ]:
all_trail_ids = activity_df[activity_df["usableTrailRun"]]["activityId"].astype(str).tolist()
all_activity_features = activity_df[activity_df["activityId"].astype(str).isin(all_trail_ids)].copy()
all_activity_features["distanceEqKm"] = pd.to_numeric(all_activity_features["distanceEqKm"], errors="coerce")
all_activity_features["distanceEqKm"] = all_activity_features["distanceEqKm"].fillna(
    pd.to_numeric(all_activity_features["distanceKm"], errors="coerce")
    + pd.to_numeric(all_activity_features["ascentM"], errors="coerce").fillna(0.0) * 0.01
)
all_activity_features["logActualTimeSec"] = np.log(all_activity_features["actualTimeSec"].clip(lower=1.0))
all_activity_features["logDistanceEqKm"] = np.log(all_activity_features["distanceEqKm"].clip(lower=1e-3))
all_activity_features["ascentPerKm"] = pd.to_numeric(all_activity_features["ascentM"], errors="coerce").fillna(0.0) / pd.to_numeric(all_activity_features["distanceKm"], errors="coerce").clip(lower=1e-3)
all_activity_features["technicalityCombined"] = pd.to_numeric(all_activity_features.get("technicalityGps"), errors="coerce")

hr_global_features = [
    "logDistanceEqKm",
    "ascentPerKm",
    "hrReserveRatio",
    "ctl",
    "tsb",
    "trimpRediSlow",
    "trimpRediBalance",
    "technicalityCombined",
    "meanAltitudeM",
    "temperatureC",
]
hr_global_rows = []
for held_out in all_trail_ids:
    train = all_activity_features[all_activity_features["activityId"].ne(held_out)]
    test = all_activity_features[all_activity_features["activityId"].eq(held_out)]
    if train.empty or test.empty:
        continue
    fitted = tpm.fit_linear_regression(train, hr_global_features, "logActualTimeSec")
    predicted = float(np.exp(tpm.predict_linear_regression(test, fitted))[0])
    actual = float(test["actualTimeSec"].iloc[0])
    hr_global_rows.append({"activityId": held_out, "actualTimeSec": actual, "predictedTimeSec": predicted})
hr_global_loo = pd.DataFrame(hr_global_rows)

all_trail_segments = segment_features[segment_features["activityId"].isin(all_trail_ids)].copy()
hr_segment_features = progressive_feature_sets[LINEAR_FINAL_STAGE]
hr_segment_rows = []
for held_out in all_trail_ids:
    train = all_trail_segments[all_trail_segments["activityId"].ne(held_out)]
    test = all_trail_segments[all_trail_segments["activityId"].eq(held_out)]
    if train.empty or test.empty:
        continue
    fitted = tpm.fit_linear_regression(train, hr_segment_features, "logActualTimeSec")
    predicted = float(np.exp(tpm.predict_linear_regression(test, fitted)).sum())
    actual = float(all_activity_features.set_index("activityId").loc[held_out, "actualTimeSec"])
    hr_segment_rows.append({"activityId": held_out, "actualTimeSec": actual, "predictedTimeSec": predicted})
hr_segment_loo = pd.DataFrame(hr_segment_rows)

hr_rows = []
for cohort_name, cohort_df in cohorts.items():
    ids = cohort_df["activityId"].astype(str).tolist()
    global_eval = hr_global_loo[hr_global_loo["activityId"].isin(ids)]
    segment_eval = hr_segment_loo[hr_segment_loo["activityId"].isin(ids)]
    if not global_eval.empty:
        hr_rows.append(metrics_row(cohort_name, "HR global LOO", global_eval["actualTimeSec"], global_eval["predictedTimeSec"]))
    if not segment_eval.empty:
        hr_rows.append(metrics_row(cohort_name, "HR E6 segment LOO", segment_eval["actualTimeSec"], segment_eval["predictedTimeSec"]))
hr_comparison = pd.DataFrame(hr_rows)
display(hr_comparison.sort_values(["cohort", "maeMin"]))


## Paper-Based Extension Stages

The staged extension keeps the Sensors speed equation and changes only one assumption at a time.

Stage 0 is the reproduction Stage 3 model:

$$\hat t_{rs}=\frac{d_{rs}3600f_{GAP,rs}}{v_{VT2}\alpha f_{alt,rs}f_{CTL,r}f_{pad}(p_{rs})}$$

Stage 1 replaces progress fatigue by acute TRIMP fatigue, without an HR speed multiplier:

$$\hat t_{rs}=\frac{d_{rs}3600f_{GAP,rs}}{VMA\alpha f_{alt,rs}f_{CTL,r}F(D_{rs})}$$

Stage 2 replaces CTL readiness with REDI readiness:

$$\hat t_{rs}=\frac{d_{rs}3600f_{GAP,rs}}{VMA\alpha f_{alt,rs}f_{REDI,r}F(D_{rs})}$$

Stage 3 adds the fixed linear HRR speed ratio:

$$\hat t_{rs}=\frac{d_{rs}3600f_{GAP,rs}}{VMA\alpha f_{alt,rs}f_{REDI,r}E(HRR_{rs})F(D_{rs})}$$

with $E(HRR)=\mathrm{clip}(HRR/0.70,E_{min},E_{max})$. $F(D)$ is grid-selected as either linear or exponential. All stages are evaluated in-sample and with cohort-level leave-one-activity-out validation on the same three cohorts.

In [ ]:
PAPER_EXT_STAGE0_ALPHA_GRID = np.round(np.arange(0.40, 1.0001, 0.05), 2)
PAPER_EXT_STAGE0_MU_GRID = np.round(np.arange(-0.50, 0.0001, 0.05), 2)
PAPER_EXT_ALPHA_GRID = np.round(np.arange(0.35, 0.851, 0.05), 2)
PAPER_EXT_KAPPA_GRID = [0.0, 0.10, 0.20, 0.30, 0.40]
PAPER_EXT_FATIGUE_MODELS = ("linear", "exponential")
PAPER_EXT_TRIMP_SCALE = 10.0
HRR_REFERENCE = 0.70


PAPER_EXT_STAGE3_FATIGUE_STATES = [
    {
        "fatigueState": "decayed",
        "acuteTrimpCol": "decayedTrimpBefore",
        "label": "decayed TRIMP",
    },
    {
        "fatigueState": "cumulative",
        "acuteTrimpCol": "cumTrimpBefore",
        "label": "cumulative TRIMP",
    },
]


def prediction_frame(
    cohort_name: str,
    model_name: str,
    cohort_df: pd.DataFrame,
    actual: pd.Series,
    predicted: pd.Series,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        {
            "activityId": actual.index.astype(str),
            "actualTimeSec": actual.to_numpy(dtype=float),
            "predictedTimeSec": predicted.reindex(actual.index).to_numpy(dtype=float),
        }
    )
    meta_cols = [
        col
        for col in ["activityId", "startDate", "name", "distanceKm", "ascentM", "hrReserveRatio"]
        if col in cohort_df.columns
    ]
    frame = frame.merge(cohort_df[meta_cols].astype({"activityId": str}), on="activityId", how="left")
    frame["cohort"] = cohort_name
    frame["model"] = model_name
    frame["actualMin"] = frame["actualTimeSec"] / 60.0
    frame["predictedMin"] = frame["predictedTimeSec"] / 60.0
    frame["errorMin"] = (frame["predictedTimeSec"] - frame["actualTimeSec"]) / 60.0
    return frame


def cohort_inputs(cohort_df: pd.DataFrame):
    ids = cohort_df["activityId"].astype(str).tolist()
    segments = {activity_id: segments_by_activity[activity_id] for activity_id in ids}
    observed = cohort_df.set_index("activityId").loc[ids, "actualTimeSec"].astype(float)
    ctl_factors = cohort_df.set_index("activityId").loc[ids, "ctlReadinessFactor"].astype(float).to_dict()
    redi_factors = cohort_df.set_index("activityId").loc[ids, "rediReadinessFactor"].astype(float).to_dict()
    return ids, segments, observed, ctl_factors, redi_factors


def race_prediction_from_segments(prediction: pd.DataFrame, ids: list[str]) -> pd.Series:
    return prediction.groupby("activityId")["predictedTimeSec"].sum().reindex(ids)


def add_stage3_variant_metadata(
    row: dict[str, object],
    *,
    validation: str,
    fatigue_state: str,
    acute_trimp_col: str,
    fatigue_model: str,
    alpha: float,
    fatigue_coef: float,
) -> dict[str, object]:
    row.update(
        {
            "validation": validation,
            "fatigueState": fatigue_state,
            "acuteTrimpCol": acute_trimp_col,
            "fatigueModel": fatigue_model,
            "alpha": alpha,
            "fatigueCoef": fatigue_coef,
        }
    )
    return row


paper_extension_stage_specs = [
    {
        "stage": "Stage 1 TRIMP fatigue CTL",
        "loadFactorCol": "ctlReadinessFactor",
        "useHrrEffort": False,
        "fatigueState": "decayed",
        "acuteTrimpCol": "decayedTrimpBefore",
    },
    {
        "stage": "Stage 2 TRIMP fatigue REDI",
        "loadFactorCol": "rediReadinessFactor",
        "useHrrEffort": False,
        "fatigueState": "decayed",
        "acuteTrimpCol": "decayedTrimpBefore",
    },
]

paper_extension_rows = []
paper_extension_parameter_rows = []
paper_extension_prediction_frames = []
paper_extension_loo_frames = []
stage3_fatigue_rows = []
stage3_best_by_cohort = {}
stage3_segments_by_cohort = {}
stage3_selection_by_cohort = {}

for cohort_name, cohort_df in cohorts.items():
    ids, segments, actual, ctl_factors, _redi_factors = cohort_inputs(cohort_df)
    if not ids:
        continue
    observed = actual.to_dict()
    cohort_segments = segment_features[segment_features["activityId"].isin(ids)].copy()
    if cohort_segments.empty:
        continue

    best_stage0, _grid_stage0 = tpm.grid_search_model(
        segments,
        observed,
        v_vt2_kmh=V_VT2_KMH,
        alpha_grid=PAPER_EXT_STAGE0_ALPHA_GRID,
        mu_grid=PAPER_EXT_STAGE0_MU_GRID,
        fatigue_model="linear",
        ctl_factors=ctl_factors,
    )
    pred_stage0 = tpm.predict_many(
        segments,
        v_vt2_kmh=V_VT2_KMH,
        alpha=best_stage0["alpha"],
        mu=best_stage0["mu"],
        fatigue_model="linear",
        ctl_factors=ctl_factors,
    ).reindex(ids)
    stage_name = "Stage 0 reproduction Stage 3"
    paper_extension_rows.append(metrics_row(cohort_name, stage_name, actual, pred_stage0))
    paper_extension_parameter_rows.append({"cohort": cohort_name, "stage": stage_name, **best_stage0})
    paper_extension_prediction_frames.append(
        prediction_frame(cohort_name, stage_name, cohort_df, actual, pred_stage0)
    )
    loo_stage0 = tpm.leave_one_out_grid_search(
        segments,
        observed,
        v_vt2_kmh=V_VT2_KMH,
        alpha_grid=PAPER_EXT_STAGE0_ALPHA_GRID,
        mu_grid=PAPER_EXT_STAGE0_MU_GRID,
        fatigue_model="linear",
        ctl_factors=ctl_factors,
    )
    if not loo_stage0.empty:
        loo_stage0["cohort"] = cohort_name
        loo_stage0["stage"] = f"{stage_name} LOO"
        paper_extension_loo_frames.append(loo_stage0)
        paper_extension_rows.append(
            metrics_row(
                cohort_name,
                f"{stage_name} LOO",
                loo_stage0["actualTimeSec"],
                loo_stage0["predictedTimeSec"],
            )
        )

    for spec in paper_extension_stage_specs:
        best, _grid, prediction = tpm.hrr_trimp_grid_search_model(
            cohort_segments,
            v_anchor_kmh=VMA_FLAT_KMH,
            alpha_grid=PAPER_EXT_ALPHA_GRID,
            fatigue_coef_grid=PAPER_EXT_KAPPA_GRID,
            fatigue_models=PAPER_EXT_FATIGUE_MODELS,
            hrr_reference=HRR_REFERENCE,
            trimp_scale=PAPER_EXT_TRIMP_SCALE,
            load_factor_col=spec["loadFactorCol"],
            use_hrr_effort=spec["useHrrEffort"],
            acute_trimp_col=spec["acuteTrimpCol"],
            objective="race",
            observed_activity_times_sec=observed,
        )
        predicted_activity = race_prediction_from_segments(prediction, ids)
        paper_extension_rows.append(metrics_row(cohort_name, spec["stage"], actual, predicted_activity))
        paper_extension_parameter_rows.append({"cohort": cohort_name, **spec, **best})
        paper_extension_prediction_frames.append(
            prediction_frame(cohort_name, spec["stage"], cohort_df, actual, predicted_activity)
        )

        loo = tpm.leave_one_out_hrr_trimp_grid_search(
            cohort_segments,
            v_anchor_kmh=VMA_FLAT_KMH,
            alpha_grid=PAPER_EXT_ALPHA_GRID,
            fatigue_coef_grid=PAPER_EXT_KAPPA_GRID,
            fatigue_models=PAPER_EXT_FATIGUE_MODELS,
            hrr_reference=HRR_REFERENCE,
            trimp_scale=PAPER_EXT_TRIMP_SCALE,
            load_factor_col=spec["loadFactorCol"],
            use_hrr_effort=spec["useHrrEffort"],
            acute_trimp_col=spec["acuteTrimpCol"],
            observed_activity_times_sec=observed,
        )
        if not loo.empty:
            loo["cohort"] = cohort_name
            loo["stage"] = f"{spec['stage']} LOO"
            loo["fatigueState"] = spec["fatigueState"]
            loo["acuteTrimpCol"] = spec["acuteTrimpCol"]
            paper_extension_loo_frames.append(loo)
            paper_extension_rows.append(
                metrics_row(
                    cohort_name,
                    f"{spec['stage']} LOO",
                    loo["actualTimeSec"],
                    loo["predictedTimeSec"],
                )
            )

    stage3_variant_results: dict[tuple[str, str], dict[str, object]] = {}
    local_stage3_rows: list[dict[str, object]] = []
    for state_spec in PAPER_EXT_STAGE3_FATIGUE_STATES:
        for fatigue_model in PAPER_EXT_FATIGUE_MODELS:
            best, _grid, prediction = tpm.hrr_trimp_grid_search_model(
                cohort_segments,
                v_anchor_kmh=VMA_FLAT_KMH,
                alpha_grid=PAPER_EXT_ALPHA_GRID,
                fatigue_coef_grid=PAPER_EXT_KAPPA_GRID,
                fatigue_models=(fatigue_model,),
                hrr_reference=HRR_REFERENCE,
                trimp_scale=PAPER_EXT_TRIMP_SCALE,
                load_factor_col="rediReadinessFactor",
                use_hrr_effort=True,
                acute_trimp_col=state_spec["acuteTrimpCol"],
                objective="race",
                observed_activity_times_sec=observed,
            )
            predicted_activity = race_prediction_from_segments(prediction, ids)
            variant_stage = f"Stage 3 HRR speed ratio {state_spec['label']} {fatigue_model}"
            in_sample_row = add_stage3_variant_metadata(
                metrics_row(cohort_name, variant_stage, actual, predicted_activity),
                validation="in_sample",
                fatigue_state=state_spec["fatigueState"],
                acute_trimp_col=state_spec["acuteTrimpCol"],
                fatigue_model=fatigue_model,
                alpha=float(best.get("alpha", np.nan)),
                fatigue_coef=float(best.get("fatigueCoef", np.nan)),
            )
            stage3_fatigue_rows.append(in_sample_row)
            local_stage3_rows.append(in_sample_row)

            loo = tpm.leave_one_out_hrr_trimp_grid_search(
                cohort_segments,
                v_anchor_kmh=VMA_FLAT_KMH,
                alpha_grid=PAPER_EXT_ALPHA_GRID,
                fatigue_coef_grid=PAPER_EXT_KAPPA_GRID,
                fatigue_models=(fatigue_model,),
                hrr_reference=HRR_REFERENCE,
                trimp_scale=PAPER_EXT_TRIMP_SCALE,
                load_factor_col="rediReadinessFactor",
                use_hrr_effort=True,
                acute_trimp_col=state_spec["acuteTrimpCol"],
                observed_activity_times_sec=observed,
            )
            if not loo.empty:
                loo["cohort"] = cohort_name
                loo["stage"] = f"{variant_stage} LOO"
                loo["fatigueState"] = state_spec["fatigueState"]
                loo["acuteTrimpCol"] = state_spec["acuteTrimpCol"]
                loo_row = add_stage3_variant_metadata(
                    metrics_row(cohort_name, f"{variant_stage} LOO", loo["actualTimeSec"], loo["predictedTimeSec"]),
                    validation="loo",
                    fatigue_state=state_spec["fatigueState"],
                    acute_trimp_col=state_spec["acuteTrimpCol"],
                    fatigue_model=fatigue_model,
                    alpha=float(pd.to_numeric(loo["alpha"], errors="coerce").median()),
                    fatigue_coef=float(pd.to_numeric(loo["fatigueCoef"], errors="coerce").median()),
                )
                stage3_fatigue_rows.append(loo_row)
                local_stage3_rows.append(loo_row)

            stage3_variant_results[(state_spec["fatigueState"], fatigue_model)] = {
                "best": {
                    **best,
                    "fatigueState": state_spec["fatigueState"],
                    "acuteTrimpCol": state_spec["acuteTrimpCol"],
                },
                "prediction": prediction,
                "predicted_activity": predicted_activity,
                "loo": loo,
            }

    local_stage3_metrics = pd.DataFrame(local_stage3_rows)
    selection_pool = local_stage3_metrics[local_stage3_metrics["validation"].eq("loo")]
    if selection_pool.empty:
        selection_pool = local_stage3_metrics[local_stage3_metrics["validation"].eq("in_sample")]
    chosen_row = selection_pool.sort_values(
        ["maeMin", "mapePct", "fatigueState", "fatigueModel"],
        ascending=[True, True, True, True],
    ).iloc[0]
    chosen_key = (str(chosen_row["fatigueState"]), str(chosen_row["fatigueModel"]))
    chosen_result = stage3_variant_results[chosen_key]
    chosen_best = dict(chosen_result["best"])
    stage3_best_by_cohort[cohort_name] = chosen_best
    stage3_segments_by_cohort[cohort_name] = cohort_segments
    stage3_selection_by_cohort[cohort_name] = chosen_row.to_dict()

    stage3_stage_name = "Stage 3 HRR speed ratio"
    paper_extension_rows.append(
        metrics_row(cohort_name, stage3_stage_name, actual, chosen_result["predicted_activity"])
    )
    paper_extension_parameter_rows.append(
        {
            "cohort": cohort_name,
            "stage": stage3_stage_name,
            "loadFactorCol": "rediReadinessFactor",
            "useHrrEffort": True,
            "fatigueState": chosen_best["fatigueState"],
            "acuteTrimpCol": chosen_best["acuteTrimpCol"],
            **chosen_best,
        }
    )
    paper_extension_prediction_frames.append(
        prediction_frame(cohort_name, stage3_stage_name, cohort_df, actual, chosen_result["predicted_activity"])
    )
    chosen_loo = chosen_result["loo"]
    if not chosen_loo.empty:
        chosen_loo = chosen_loo.copy()
        chosen_loo["cohort"] = cohort_name
        chosen_loo["stage"] = f"{stage3_stage_name} LOO"
        chosen_loo["fatigueState"] = chosen_best["fatigueState"]
        chosen_loo["acuteTrimpCol"] = chosen_best["acuteTrimpCol"]
        paper_extension_loo_frames.append(chosen_loo)
        paper_extension_rows.append(
            metrics_row(
                cohort_name,
                f"{stage3_stage_name} LOO",
                chosen_loo["actualTimeSec"],
                chosen_loo["predictedTimeSec"],
            )
        )

paper_extension_comparison = pd.DataFrame(paper_extension_rows)
paper_extension_parameters = pd.DataFrame(paper_extension_parameter_rows)
stage3_fatigue_comparison = pd.DataFrame(stage3_fatigue_rows)
paper_extension_predictions = (
    pd.concat(paper_extension_prediction_frames, ignore_index=True)
    if paper_extension_prediction_frames
    else pd.DataFrame()
)
paper_extension_loo = (
    pd.concat(paper_extension_loo_frames, ignore_index=True)
    if paper_extension_loo_frames
    else pd.DataFrame()
)

ablation_rows = []
for cohort_name, best in stage3_best_by_cohort.items():
    cohort_df = cohorts[cohort_name]
    ids = cohort_df["activityId"].astype(str).tolist()
    actual = cohort_df.set_index("activityId").loc[ids, "actualTimeSec"].astype(float)
    base_segments = stage3_segments_by_cohort[cohort_name]
    acute_trimp_col = str(best.get("acuteTrimpCol", "decayedTrimpBefore"))
    no_gap_segments = base_segments.assign(avgGrade=0.0).copy()
    for gap_col in ["gapFactorIntegrated", "gapFactor"]:
        if gap_col in no_gap_segments.columns:
            no_gap_segments[gap_col] = 1.0
    variants = {
        "full": (base_segments.copy(), {"use_hrr_effort": True, "fatigue_coef": float(best["fatigueCoef"]), "load_factor_col": "rediReadinessFactor"}),
        "no GAP": (no_gap_segments, {"use_hrr_effort": True, "fatigue_coef": float(best["fatigueCoef"]), "load_factor_col": "rediReadinessFactor"}),
        "no altitude": (base_segments.assign(meanAltitudeM=0.0), {"use_hrr_effort": True, "fatigue_coef": float(best["fatigueCoef"]), "load_factor_col": "rediReadinessFactor"}),
        "no REDI readiness": (base_segments.assign(rediReadinessFactor=1.0), {"use_hrr_effort": True, "fatigue_coef": float(best["fatigueCoef"]), "load_factor_col": "rediReadinessFactor"}),
        "no HRR speed ratio": (base_segments.copy(), {"use_hrr_effort": False, "fatigue_coef": float(best["fatigueCoef"]), "load_factor_col": "rediReadinessFactor"}),
        "no TRIMP fatigue": (base_segments.copy(), {"use_hrr_effort": True, "fatigue_coef": 0.0, "load_factor_col": "rediReadinessFactor"}),
    }
    full_mae = None
    for variant_name, (variant_segments, kwargs) in variants.items():
        predicted_segments = tpm.predict_hrr_trimp_segment_times(
            variant_segments,
            v_anchor_kmh=VMA_FLAT_KMH,
            alpha=float(best["alpha"]),
            fatigue_model=str(best["fatigueModel"]),
            hrr_reference=HRR_REFERENCE,
            trimp_scale=PAPER_EXT_TRIMP_SCALE,
            acute_trimp_col=acute_trimp_col,
            **kwargs,
        )
        predicted_activity = (
            pd.DataFrame({"activityId": variant_segments["activityId"], "predicted": predicted_segments})
            .groupby("activityId")["predicted"]
            .sum()
            .reindex(ids)
        )
        row = metrics_row(cohort_name, variant_name, actual, predicted_activity)
        row["fatigueState"] = best.get("fatigueState", "")
        row["acuteTrimpCol"] = acute_trimp_col
        row["fatigueModel"] = best.get("fatigueModel", "")
        if variant_name == "full":
            full_mae = row["maeMin"]
        row["deltaMaeMinVsFull"] = row["maeMin"] - full_mae if full_mae is not None else 0.0
        ablation_rows.append(row)
paper_stage_ablation = pd.DataFrame(ablation_rows)

validation_rows = []
if not linear_loo.empty:
    validation_rows.append(
        extension_comparison[extension_comparison["stage"].str.contains("LOO", regex=False)]
    )
if not hr_comparison.empty:
    validation_rows.append(hr_comparison)
if not paper_extension_comparison.empty:
    validation_rows.append(
        paper_extension_comparison[paper_extension_comparison["stage"].str.contains("LOO", regex=False)]
    )
validation_comparison = (
    pd.concat(validation_rows, ignore_index=True)
    if validation_rows
    else pd.DataFrame()
)

parameter_cols = [
    col for col in [
        "cohort",
        "stage",
        "fatigueState",
        "acuteTrimpCol",
        "alpha",
        "mu",
        "fatigueModel",
        "fatigueCoef",
        "r2",
        "maeSec",
        "raceR2",
        "raceMaeSec",
    ]
    if col in paper_extension_parameters.columns
]
display(paper_extension_parameters[parameter_cols])
display(stage3_fatigue_comparison.sort_values(["cohort", "validation", "maeMin"]))
display(paper_extension_comparison.sort_values(["cohort", "maeMin"]))
display(paper_stage_ablation.sort_values(["cohort", "deltaMaeMinVsFull"], ascending=[True, False]))
display(validation_comparison.sort_values(["cohort", "maeMin"]))

if not paper_extension_predictions.empty:
    fig, axes = plt.subplots(1, len(cohorts), figsize=(5 * len(cohorts), 4), squeeze=False)
    for ax, (cohort_name, _cohort_df) in zip(axes[0], cohorts.items()):
        subset = paper_extension_predictions[paper_extension_predictions["cohort"].eq(cohort_name)]
        if subset.empty:
            ax.set_axis_off()
            continue
        limit = max(subset["actualMin"].max(), subset["predictedMin"].max()) * 1.08
        for model_name, model_df in subset.groupby("model", sort=False):
            ax.scatter(model_df["actualMin"], model_df["predictedMin"], alpha=0.65, label=model_name)
        ax.plot([0, limit], [0, limit], "k--")
        ax.fill_between([0, limit], [0, limit * 0.9], [0, limit * 1.1], color="gray", alpha=0.12)
        ax.set_title(cohort_name)
        ax.set_xlabel("Actual time (min)")
        ax.set_ylabel("Predicted time (min)")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)
    fig.suptitle("Paper-based extension in-sample stage comparison")
    fig.tight_layout()
    plt.show()




## Exploratory Segment-Level Grid Search

This diagnostic is retained for comparison only. It is not part of the staged paper-based extension above because it optimizes many segment modifiers at once.

$$\hat t_s = \frac{d_s\,3600\,f_{GAP,s}}{VMA\,\alpha\,f_{alt,s}\,f_{fatigue}(s)\,M_{terrain}\,\exp(\gamma_{HR}(HRR_s-0.70)-\gamma_{tech}T_s-\gamma_{acute}U_s)}$$

Both segment metrics and race-summed metrics are reported to detect segment overfit that degrades finish-time prediction.

In [ ]:
SEGMENT_ALPHA_GRID = np.round(np.arange(0.30, 0.751, 0.05), 2)
SEGMENT_MU_GRID = [-0.30, -0.15, 0.0]
TERRAIN_MULTIPLIER_GRID = [
    {"steep_climb": 1.0, "climb": 1.0, "flat": 1.0, "descent": 1.0, "steep_descent": 1.0},
    {"steep_climb": 0.55, "climb": 0.80, "flat": 1.00, "descent": 0.90, "steep_descent": 0.65},
    {"steep_climb": 0.45, "climb": 0.75, "flat": 1.05, "descent": 0.85, "steep_descent": 0.55},
]

segment_grid_rows = []
grouped_diagnostic_frames = []
segment_prediction_frames = []
for cohort_name, cohort_df in cohorts.items():
    ids = cohort_df["activityId"].astype(str).tolist()
    cohort_segments = segment_features[segment_features["activityId"].isin(ids)].copy()
    if cohort_segments.empty:
        continue
    best, grid, prediction = tpm.segment_grid_search_model(
        cohort_segments,
        v_anchor_kmh=VMA_FLAT_KMH,
        alpha_grid=SEGMENT_ALPHA_GRID,
        mu_grid=SEGMENT_MU_GRID,
        terrain_multiplier_grid=TERRAIN_MULTIPLIER_GRID,
        technicality_coef_grid=[0.0, 0.25],
        hr_coef_grid=[0.0, 0.35],
        acute_trimp_coef_grid=[0.0, 0.04],
    )
    segment_grid_rows.append({"cohort": cohort_name, **best})
    prediction["cohort"] = cohort_name
    segment_prediction_frames.append(prediction)
    grouped = tpm.grouped_segment_metrics(prediction, group_col="terrainFamily")
    grouped["cohort"] = cohort_name
    grouped_diagnostic_frames.append(grouped)

segment_grid_comparison = pd.DataFrame(segment_grid_rows)
segment_grid_predictions = pd.concat(segment_prediction_frames, ignore_index=True) if segment_prediction_frames else pd.DataFrame()
terrain_group_diagnostics = pd.concat(grouped_diagnostic_frames, ignore_index=True) if grouped_diagnostic_frames else pd.DataFrame()

display(segment_grid_comparison[[
    "cohort",
    "alpha",
    "mu",
    "terrainProfile",
    "technicalityCoef",
    "hrCoef",
    "acuteTrimpCoef",
    "segmentR2",
    "segmentMaeSec",
    "segmentMapePct",
    "raceR2",
    "raceMaeSec",
    "raceMapePct",
]])
display(terrain_group_diagnostics)

hard_pred = segment_grid_predictions[segment_grid_predictions["cohort"].eq("hardTrailRun")]
if not hard_pred.empty:
    fig, ax = plt.subplots()
    ax.scatter(hard_pred["actualTimeSec"] / 60.0, hard_pred["predictedTimeSec"] / 60.0, alpha=0.45)
    limit = max(hard_pred["actualTimeSec"].max(), hard_pred["predictedTimeSec"].max()) / 60.0
    ax.plot([0, limit], [0, limit], "k--")
    ax.set_xlabel("Actual segment time (min)")
    ax.set_ylabel("Predicted segment time (min)")
    ax.set_title("Extension segment grid search - hard TrailRuns")
    plt.show()


## Stage 3 Fit Visual Comparison

The next plots mirror the reproduction notebook visual check for the paper-based extension. The first figure compares Stage 0-3 race-level fits across the three cohorts. The second figure repeats the same race-summed view for the exploratory segment-grid diagnostic.

In [ ]:
def plot_fit_comparison(predictions: pd.DataFrame, title: str, model_order: list[str] | None = None) -> None:
    if predictions.empty:
        print(f"No predictions available for {title}")
        return

    cohorts_order = [
        name
        for name in ["hardTrailRun", "top10HardTrailByHRR", "selectedDateRaces"]
        if name in predictions["cohort"].unique()
    ]
    if model_order is None:
        models = predictions["model"].drop_duplicates().tolist()
    else:
        models = [model for model in model_order if model in predictions["model"].unique()]
    if not cohorts_order or not models:
        print(f"No plottable predictions available for {title}")
        return

    colors = dict(zip(models, plt.cm.tab10.colors[: len(models)]))
    fig, axes = plt.subplots(len(cohorts_order), 2, figsize=(14, 4.2 * len(cohorts_order)), squeeze=False)
    for row_idx, cohort_name in enumerate(cohorts_order):
        cohort_pred = predictions[predictions["cohort"].eq(cohort_name)].copy()
        cohort_pred = cohort_pred[cohort_pred["model"].isin(models)]
        if cohort_pred.empty:
            axes[row_idx, 0].set_axis_off()
            axes[row_idx, 1].set_axis_off()
            continue
        max_minutes = float(np.nanmax([cohort_pred["actualMin"].max(), cohort_pred["predictedMin"].max()]))
        limit = max(max_minutes * 1.08, 1.0)

        ax = axes[row_idx, 0]
        for model_name in models:
            model_df = cohort_pred[cohort_pred["model"].eq(model_name)]
            if model_df.empty:
                continue
            ax.scatter(
                model_df["actualMin"],
                model_df["predictedMin"],
                label=model_name,
                alpha=0.78,
                s=46,
                color=colors.get(model_name),
            )
        ax.plot([0, limit], [0, limit], color="black", linewidth=1, label="Perfect fit")
        ax.fill_between([0, limit], [0, limit * 0.9], [0, limit * 1.1], color="gray", alpha=0.12, label="+/-10%")
        ax.set_xlim(0, limit)
        ax.set_ylim(0, limit)
        ax.set_xlabel("Actual race time (min)")
        ax.set_ylabel("Predicted race time (min)")
        ax.set_title(f"{cohort_name}: predicted vs actual")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[row_idx, 1]
        residual_data = [cohort_pred[cohort_pred["model"].eq(model)]["errorMin"].dropna().to_numpy() for model in models]
        ax.boxplot(residual_data, tick_labels=models, showmeans=True)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_ylabel("Prediction error (min)")
        ax.set_title(f"{cohort_name}: residual distribution")
        ax.tick_params(axis="x", rotation=25)
        ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(title, y=1.01, fontsize=14)
    fig.tight_layout()
    plt.show()


EXTENSION_STAGE_ORDER = [
    "Stage 0 reproduction Stage 3",
    "Stage 1 TRIMP fatigue CTL",
    "Stage 2 TRIMP fatigue REDI",
    "Stage 3 HRR speed ratio",
]

plot_fit_comparison(
    paper_extension_predictions,
    "Paper-based extension Stage 0-3 race-level fit comparison",
    model_order=EXTENSION_STAGE_ORDER,
)

if "segment_grid_predictions" in globals() and not segment_grid_predictions.empty:
    segment_grid_race_predictions = (
        segment_grid_predictions.groupby(["cohort", "activityId"], as_index=False)
        .agg(actualTimeSec=("actualTimeSec", "sum"), predictedTimeSec=("predictedTimeSec", "sum"))
    )
    segment_grid_race_predictions["model"] = "Segment-grid diagnostic"
    segment_grid_race_predictions["actualMin"] = segment_grid_race_predictions["actualTimeSec"] / 60.0
    segment_grid_race_predictions["predictedMin"] = segment_grid_race_predictions["predictedTimeSec"] / 60.0
    segment_grid_race_predictions["errorMin"] = (
        segment_grid_race_predictions["predictedTimeSec"] - segment_grid_race_predictions["actualTimeSec"]
    ) / 60.0
    plot_fit_comparison(
        segment_grid_race_predictions,
        "Race-summed exploratory segment-grid fit comparison",
    )


## Interpretation

Use the paper-based Stage 0-3 table as the primary extension result. The linear E1-E6bis regression is a coefficient diagnostic, and the segment grid search is an exploratory stress test. A useful extension should improve LOO race error, not only in-sample or segment-level error.

## Paper Assets

This section exports manuscript-ready figures, result tables, robustness checks, and anonymized derived feature files under `docs/science/paper_assets/`. The exports are regenerated from notebook variables so manuscript numbers remain reproducible.

In [ ]:
PAPER_ASSET_DIR = PROJECT_ROOT / "docs" / "science" / "paper_assets"
PAPER_ASSET_DIR.mkdir(parents=True, exist_ok=True)


def save_paper_figure(fig: plt.Figure, filename: str) -> Path:
    path = PAPER_ASSET_DIR / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def write_markdown_table(df: pd.DataFrame, filename: str) -> Path:
    path = PAPER_ASSET_DIR / filename
    if df.empty:
        path.write_text("_No rows available._\n")
        return path
    text_df = df.fillna("").astype(str)
    headers = text_df.columns.tolist()
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for _, row in text_df.iterrows():
        values = [str(row[col]).replace("|", "\\|") for col in headers]
        lines.append("| " + " | ".join(values) + " |")
    path.write_text("\n".join(lines) + "\n")
    return path


def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = PAPER_ASSET_DIR / filename
    df.to_csv(path, index=False)
    return path


def cohort_descriptive_rows() -> pd.DataFrame:
    rows = []
    for cohort_name, cohort_df in cohorts.items():
        working = cohort_df.copy()
        rows.append(
            {
                "cohort": cohort_name,
                "activityCount": int(len(working)),
                "distanceKmTotal": pd.to_numeric(working.get("distanceKm"), errors="coerce").sum(),
                "distanceKmMedian": pd.to_numeric(working.get("distanceKm"), errors="coerce").median(),
                "ascentMTotal": pd.to_numeric(working.get("ascentM"), errors="coerce").sum(),
                "ascentMMedian": pd.to_numeric(working.get("ascentM"), errors="coerce").median(),
                "durationMinMedian": pd.to_numeric(working.get("actualTimeSec"), errors="coerce").median() / 60.0,
                "durationMinTotal": pd.to_numeric(working.get("actualTimeSec"), errors="coerce").sum() / 60.0,
                "hrrMean": pd.to_numeric(working.get("hrReserveRatio"), errors="coerce").mean(),
                "hrrMedian": pd.to_numeric(working.get("hrReserveRatio"), errors="coerce").median(),
                "ctlMedian": pd.to_numeric(working.get("ctl"), errors="coerce").median(),
                "tsbMedian": pd.to_numeric(working.get("tsb"), errors="coerce").median(),
                "rediSlowMedian": pd.to_numeric(working.get("trimpRediSlow"), errors="coerce").median(),
                "rediBalanceMedian": pd.to_numeric(working.get("trimpRediBalance"), errors="coerce").median(),
            }
        )
    return pd.DataFrame(rows)


MODEL_EQUATION_ROWS = pd.DataFrame(
    [
        {
            "stage": "Stage 0",
            "description": "Sensors-style reproduction Stage 3 with CTL readiness and progress fatigue",
            "equation": r"t = d 3600 f_GAP / (v_VT2 alpha f_alt f_CTL f_pad(p))",
        },
        {
            "stage": "Stage 1",
            "description": "Replace progress fatigue with decayed acute TRIMP fatigue",
            "equation": r"t = d 3600 f_GAP / (VMA alpha f_alt f_CTL F(D))",
        },
        {
            "stage": "Stage 2",
            "description": "Replace CTL readiness with REDI readiness",
            "equation": r"t = d 3600 f_GAP / (VMA alpha f_alt f_REDI F(D))",
        },
        {
            "stage": "Stage 3",
            "description": "Add fixed linear HRR speed ratio and select decayed or cumulative acute TRIMP fatigue",
            "equation": r"t = d 3600 f_GAP / (VMA alpha f_alt f_REDI E(HRR) F(U)), U in {D, C}",
        },
        {
            "stage": "Full regression comparator",
            "description": "OLS log-time diagnostic with E6 decayed TRIMP or E6bis cumulative TRIMP",
            "equation": r"log(t) = beta0 + beta_d log(d) + beta_g log(f_GAP) + ...",
        },
    ]
)

paper_asset_paths: list[Path] = []
paper_asset_paths.append(save_table(cohort_descriptive_rows(), "table_cohort_descriptives.csv"))
paper_asset_paths.append(write_markdown_table(MODEL_EQUATION_ROWS, "table_model_equations.md"))
paper_asset_paths.append(save_table(paper_extension_comparison, "table_stage_metrics.csv"))
paper_asset_paths.append(save_table(paper_extension_parameters, "table_fitted_parameters.csv"))
paper_asset_paths.append(save_table(stage3_fatigue_comparison, "table_stage3_fatigue_state_comparison.csv"))
paper_asset_paths.append(save_table(hr_comparison, "table_hr_regression_metrics.csv"))
paper_asset_paths.append(save_table(linear_variable_importance, "table_full_regression_coefficients.csv"))
paper_asset_paths.append(save_table(paper_stage_ablation, "table_stage3_ablation.csv"))

print(f"Paper asset directory: {PAPER_ASSET_DIR}")



In [ ]:
# Manuscript figures.
STAGE_ORDER = [
    "Stage 0 reproduction Stage 3",
    "Stage 1 TRIMP fatigue CTL",
    "Stage 2 TRIMP fatigue REDI",
    "Stage 3 HRR speed ratio",
]
STAGE_COLORS = dict(zip(STAGE_ORDER, plt.cm.tab10.colors[: len(STAGE_ORDER)]))

# 1. Stage-flow schematic.
fig, ax = plt.subplots(figsize=(13, 3.6))
ax.set_axis_off()
stage_boxes = [
    ("Stage 0", "GAP + altitude\nCTL readiness\nprogress fatigue"),
    ("Stage 1", "Replace progress\nwith acute TRIMP\nfatigue F(D)"),
    ("Stage 2", "Replace CTL\nwith REDI\nreadiness"),
    ("Stage 3", "Add linear\nHRR speed ratio\nselect F(D) or F(C)"),
]
for idx, (title, body) in enumerate(stage_boxes):
    x = 0.08 + idx * 0.23
    ax.text(
        x,
        0.56,
        f"{title}\n{body}",
        ha="center",
        va="center",
        fontsize=10,
        bbox={"boxstyle": "round,pad=0.45", "facecolor": "#f4f6f8", "edgecolor": "#52616f"},
        transform=ax.transAxes,
    )
    if idx < len(stage_boxes) - 1:
        ax.annotate(
            "",
            xy=(x + 0.165, 0.56),
            xytext=(x + 0.105, 0.56),
            arrowprops={"arrowstyle": "->", "lw": 1.2, "color": "#52616f"},
            xycoords=ax.transAxes,
        )
ax.text(
    0.5,
    0.08,
    "Primary interpretation: Stage 3 is retrospective because it uses observed HRR.",
    ha="center",
    va="center",
    fontsize=10,
    transform=ax.transAxes,
)
fig.suptitle("Interpretable paper-based extension stages", fontsize=14)
paper_asset_paths.append(save_paper_figure(fig, "fig_model_stage_flow.png"))

# 2. Stage predicted vs actual.
fig, axes = plt.subplots(1, len(cohorts), figsize=(5.2 * len(cohorts), 4.3), squeeze=False)
for ax, (cohort_name, _cohort_df) in zip(axes[0], cohorts.items()):
    subset = paper_extension_predictions[
        paper_extension_predictions["cohort"].eq(cohort_name)
        & paper_extension_predictions["model"].isin(STAGE_ORDER)
    ].copy()
    if subset.empty:
        ax.set_axis_off()
        continue
    limit = max(subset["actualMin"].max(), subset["predictedMin"].max()) * 1.08
    for model_name in STAGE_ORDER:
        model_df = subset[subset["model"].eq(model_name)]
        if model_df.empty:
            continue
        ax.scatter(
            model_df["actualMin"],
            model_df["predictedMin"],
            alpha=0.72,
            label=model_name.replace("Stage ", "S"),
            color=STAGE_COLORS[model_name],
        )
    ax.plot([0, limit], [0, limit], color="black", linewidth=1)
    ax.fill_between([0, limit], [0, 0.9 * limit], [0, 1.1 * limit], color="gray", alpha=0.12)
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_title(cohort_name)
    ax.set_xlabel("Actual time (min)")
    ax.set_ylabel("Predicted time (min)")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7)
fig.suptitle("Paper-based Stage 0-3 predicted vs actual", fontsize=14)
fig.tight_layout()
paper_asset_paths.append(save_paper_figure(fig, "fig_stage_predicted_vs_actual.png"))

# 2b. Stage 3 fatigue-state comparison.
stage3_compare_plot = stage3_fatigue_comparison[
    stage3_fatigue_comparison["validation"].eq("loo")
].copy()
if stage3_compare_plot.empty:
    stage3_compare_plot = stage3_fatigue_comparison[
        stage3_fatigue_comparison["validation"].eq("in_sample")
    ].copy()
if not stage3_compare_plot.empty:
    stage3_compare_plot["variantLabel"] = (
        stage3_compare_plot["fatigueState"].astype(str)
        + "\n"
        + stage3_compare_plot["fatigueModel"].astype(str)
    )
    fig, axes = plt.subplots(1, len(cohorts), figsize=(4.9 * len(cohorts), 4.3), squeeze=False)
    for ax, cohort_name in zip(axes[0], cohorts.keys()):
        subset = stage3_compare_plot[stage3_compare_plot["cohort"].eq(cohort_name)].copy()
        if subset.empty:
            ax.set_axis_off()
            continue
        subset = subset.sort_values(["fatigueState", "fatigueModel"])
        colors = ["#4c78a8" if state == "decayed" else "#f58518" for state in subset["fatigueState"]]
        ax.bar(subset["variantLabel"], subset["maeMin"], color=colors)
        best = subset.sort_values(["maeMin", "mapePct"]).iloc[0]
        ax.axhline(float(best["maeMin"]), color="black", linewidth=1, linestyle="--")
        ax.set_title(cohort_name)
        ax.set_ylabel("LOO MAE (min)" if subset["validation"].eq("loo").any() else "MAE (min)")
        ax.tick_params(axis="x", rotation=0)
        ax.grid(True, axis="y", alpha=0.25)
    fig.suptitle("Stage 3 acute fatigue state and shape comparison", fontsize=14)
    fig.tight_layout()
    paper_asset_paths.append(save_paper_figure(fig, "fig_stage3_fatigue_state_comparison.png"))

# 3. Stage 3 LOO residuals by distance, ascent, and HRR.
stage3_loo = paper_extension_loo[paper_extension_loo["stage"].eq("Stage 3 HRR speed ratio LOO")].copy()
activity_meta = activity_df[["activityId", "distanceKm", "ascentM", "hrReserveRatio"]].astype({"activityId": str})
stage3_loo = stage3_loo.merge(activity_meta, on="activityId", how="left")
stage3_loo["errorMin"] = stage3_loo["errorSec"] / 60.0
fig, axes = plt.subplots(1, 3, figsize=(14, 4), squeeze=False)
for ax, x_col, label in zip(
    axes[0],
    ["distanceKm", "ascentM", "hrReserveRatio"],
    ["Distance (km)", "Ascent (m)", "HR reserve ratio"],
):
    for cohort_name, group in stage3_loo.groupby("cohort", sort=False):
        ax.scatter(group[x_col], group["errorMin"], alpha=0.72, label=cohort_name)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel(label)
    ax.set_ylabel("Stage 3 LOO residual (min)")
    ax.grid(True, alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.suptitle("Stage 3 LOO residual drivers", fontsize=14)
fig.tight_layout()
paper_asset_paths.append(save_paper_figure(fig, "fig_loo_residuals_drivers.png"))

# 4. Stage 3 ablation.
fig, axes = plt.subplots(1, len(cohorts), figsize=(5.0 * len(cohorts), 4.5), squeeze=False)
for ax, cohort_name in zip(axes[0], cohorts.keys()):
    subset = paper_stage_ablation[
        paper_stage_ablation["cohort"].eq(cohort_name) & paper_stage_ablation["stage"].ne("full")
    ].sort_values("deltaMaeMinVsFull")
    ax.barh(subset["stage"], subset["deltaMaeMinVsFull"], color="#4c78a8")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(cohort_name)
    ax.set_xlabel("Delta MAE vs full Stage 3 (min)")
    ax.grid(True, axis="x", alpha=0.25)
fig.suptitle("Stage 3 component ablation", fontsize=14)
fig.tight_layout()
paper_asset_paths.append(save_paper_figure(fig, "fig_stage3_ablation.png"))

# 5. Full-regression standardized coefficients.
coef_plot = linear_variable_importance[
    linear_variable_importance["stage"].isin(["E6 decayed acute TRIMP", "E6bis cumulative acute TRIMP"])
    & linear_variable_importance["feature"].ne("intercept")
].copy()
fig, axes = plt.subplots(len(cohorts), 1, figsize=(11, 4.2 * len(cohorts)), squeeze=False)
for ax, cohort_name in zip(axes[:, 0], cohorts.keys()):
    subset = coef_plot[coef_plot["cohort"].eq(cohort_name)].copy()
    subset = subset.sort_values("absStandardizedCoefficient", ascending=False).head(14)
    labels = subset["stage"].str.replace(" acute TRIMP", "", regex=False) + " | " + subset["feature"]
    ax.barh(labels[::-1], subset["standardizedCoefficient"].iloc[::-1], color="#72b7b2")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(cohort_name)
    ax.set_xlabel("Standardized coefficient")
    ax.grid(True, axis="x", alpha=0.25)
fig.suptitle("E6/E6bis standardized regression coefficients", fontsize=14)
fig.tight_layout()
paper_asset_paths.append(save_paper_figure(fig, "fig_regression_coefficients.png"))

# 6. HRR vs speed-equivalent and Stage 3 segment residuals.
stage3_segment_frames = []
for cohort_name, best in stage3_best_by_cohort.items():
    cohort_segments = stage3_segments_by_cohort[cohort_name].copy()
    predicted = tpm.predict_hrr_trimp_segment_times(
        cohort_segments,
        v_anchor_kmh=VMA_FLAT_KMH,
        alpha=float(best["alpha"]),
        fatigue_coef=float(best["fatigueCoef"]),
        fatigue_model=str(best["fatigueModel"]),
        hrr_reference=HRR_REFERENCE,
        trimp_scale=PAPER_EXT_TRIMP_SCALE,
        load_factor_col="rediReadinessFactor",
        use_hrr_effort=True,
        acute_trimp_col=str(best.get("acuteTrimpCol", "decayedTrimpBefore")),
    )
    cohort_segments["cohort"] = cohort_name
    cohort_segments["stage3FatigueState"] = best.get("fatigueState", "")
    cohort_segments["stage3AcuteTrimpCol"] = best.get("acuteTrimpCol", "")
    cohort_segments["stage3FatigueModel"] = best.get("fatigueModel", "")
    cohort_segments["stage3PredictedTimeSec"] = predicted
    cohort_segments["stage3ResidualSec"] = predicted - pd.to_numeric(cohort_segments["actualTimeSec"], errors="coerce")
    stage3_segment_frames.append(cohort_segments)
stage3_segment_predictions = pd.concat(stage3_segment_frames, ignore_index=True) if stage3_segment_frames else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), squeeze=False)
all_speed_df = segment_features.copy()
all_speed_df["speedEqKmh"] = (
    pd.to_numeric(all_speed_df["distanceKm"], errors="coerce")
    / (pd.to_numeric(all_speed_df["actualTimeSec"], errors="coerce") / 3600.0).clip(lower=1e-6)
    * pd.to_numeric(all_speed_df["gapFactor"], errors="coerce")
)
axes[0, 0].scatter(all_speed_df["meanHrReserve"], all_speed_df["speedEqKmh"], alpha=0.25, s=16)
axes[0, 0].set_xlabel("Segment HR reserve ratio")
axes[0, 0].set_ylabel("Speed-equivalent km/h")
axes[0, 0].set_title("Observed HRR vs speed-equivalent")
axes[0, 0].grid(True, alpha=0.25)
for cohort_name, group in stage3_segment_predictions.groupby("cohort", sort=False):
    axes[0, 1].scatter(
        group["meanHrReserve"],
        group["stage3ResidualSec"] / 60.0,
        alpha=0.35,
        s=18,
        label=cohort_name,
    )
axes[0, 1].axhline(0, color="black", linewidth=1)
axes[0, 1].set_xlabel("Segment HR reserve ratio")
axes[0, 1].set_ylabel("Stage 3 segment residual (min)")
axes[0, 1].set_title("Stage 3 residuals by HRR")
axes[0, 1].grid(True, alpha=0.25)
axes[0, 1].legend(fontsize=8)
fig.tight_layout()
paper_asset_paths.append(save_paper_figure(fig, "fig_hrr_speed_residual.png"))



In [ ]:
# Robustness checks for manuscript sensitivity tables and heatmaps.
ROBUSTNESS_TRIMP_SCALES = [5.0, 10.0, 20.0, 40.0]
ROBUSTNESS_HRR_REFERENCES = [0.65, 0.70, 0.75]
ROBUSTNESS_DECAY_LAMBDAS = [0.15, 0.30, 0.60]
RUN_SEGMENT_LENGTH_SENSITIVITY = False

robustness_rows = []
robustness_segment_features_by_decay: dict[float, pd.DataFrame] = {}
for decay_lambda in ROBUSTNESS_DECAY_LAMBDAS:
    decayed_segments = tpm.add_in_activity_trimp_features(all_segments_df, decay_lambda=decay_lambda)
    robustness_segment_features_by_decay[decay_lambda] = add_segment_model_features(decayed_segments, activity_df)

for decay_lambda, robust_features in robustness_segment_features_by_decay.items():
    for trimp_scale in ROBUSTNESS_TRIMP_SCALES:
        for hrr_reference in ROBUSTNESS_HRR_REFERENCES:
            for cohort_name, cohort_df in cohorts.items():
                ids = cohort_df["activityId"].astype(str).tolist()
                cohort_segments = robust_features[robust_features["activityId"].isin(ids)].copy()
                if cohort_segments.empty:
                    continue
                observed = cohort_df.set_index("activityId").loc[ids, "actualTimeSec"].astype(float).to_dict()
                best, _grid, _prediction = tpm.hrr_trimp_grid_search_model(
                    cohort_segments,
                    v_anchor_kmh=VMA_FLAT_KMH,
                    alpha_grid=PAPER_EXT_ALPHA_GRID,
                    fatigue_coef_grid=PAPER_EXT_KAPPA_GRID,
                    fatigue_models=PAPER_EXT_FATIGUE_MODELS,
                    hrr_reference=hrr_reference,
                    trimp_scale=trimp_scale,
                    load_factor_col="rediReadinessFactor",
                    use_hrr_effort=True,
                    acute_trimp_col="decayedTrimpBefore",
                    objective="race",
                    observed_activity_times_sec=observed,
                )
                robustness_rows.append(
                    {
                        "checkType": "stage3_grid_sensitivity",
                        "validation": "in_sample",
                        "cohort": cohort_name,
                        "model": "Stage 3 HRR speed ratio",
                        "trimpScale": trimp_scale,
                        "hrrReference": hrr_reference,
                        "decayLambda": decay_lambda,
                        "segmentKm": SEGMENT_KM,
                        "target": "moving_time",
                        "status": "computed",
                        "alpha": best.get("alpha", np.nan),
                        "fatigueModel": best.get("fatigueModel", ""),
                        "fatigueCoef": best.get("fatigueCoef", np.nan),
                        "r2": best.get("raceR2", np.nan),
                        "maeMin": best.get("raceMaeSec", np.nan) / 60.0,
                        "mapePct": best.get("raceMapePct", np.nan),
                        "biasMin": best.get("raceBiasSec", np.nan) / 60.0,
                    }
                )

# Target sensitivity: fit Stage 3 against elapsed time where available.
for cohort_name, cohort_df in cohorts.items():
    ids = cohort_df["activityId"].astype(str).tolist()
    cohort_segments = segment_features[segment_features["activityId"].isin(ids)].copy()
    elapsed = pd.to_numeric(cohort_df.set_index("activityId").loc[ids].get("elapsedSec"), errors="coerce")
    moving = pd.to_numeric(cohort_df.set_index("activityId").loc[ids].get("actualTimeSec"), errors="coerce")
    observed_elapsed = elapsed.where(elapsed > 0, moving).astype(float).to_dict()
    if cohort_segments.empty:
        continue
    best, _grid, _prediction = tpm.hrr_trimp_grid_search_model(
        cohort_segments,
        v_anchor_kmh=VMA_FLAT_KMH,
        alpha_grid=PAPER_EXT_ALPHA_GRID,
        fatigue_coef_grid=PAPER_EXT_KAPPA_GRID,
        fatigue_models=PAPER_EXT_FATIGUE_MODELS,
        hrr_reference=HRR_REFERENCE,
        trimp_scale=PAPER_EXT_TRIMP_SCALE,
        load_factor_col="rediReadinessFactor",
        use_hrr_effort=True,
        acute_trimp_col="decayedTrimpBefore",
        objective="race",
        observed_activity_times_sec=observed_elapsed,
    )
    robustness_rows.append(
        {
            "checkType": "target_sensitivity",
            "validation": "in_sample",
            "cohort": cohort_name,
            "model": "Stage 3 HRR speed ratio",
            "trimpScale": PAPER_EXT_TRIMP_SCALE,
            "hrrReference": HRR_REFERENCE,
            "decayLambda": 0.30,
            "segmentKm": SEGMENT_KM,
            "target": "elapsed_time",
            "status": "computed",
            "alpha": best.get("alpha", np.nan),
            "fatigueModel": best.get("fatigueModel", ""),
            "fatigueCoef": best.get("fatigueCoef", np.nan),
            "r2": best.get("raceR2", np.nan),
            "maeMin": best.get("raceMaeSec", np.nan) / 60.0,
            "mapePct": best.get("raceMapePct", np.nan),
            "biasMin": best.get("raceBiasSec", np.nan) / 60.0,
        }
    )

# Segment-length sensitivity is implemented as an optional appendix switch because it requires
# rebuilding all time-series segments. The primary manuscript exports the 1 km result.
for cohort_name in cohorts.keys():
    robustness_rows.append(
        {
            "checkType": "segment_length_sensitivity",
            "validation": "not_run_optional_appendix" if not RUN_SEGMENT_LENGTH_SENSITIVITY else "computed",
            "cohort": cohort_name,
            "model": "Stage 3 HRR speed ratio",
            "trimpScale": PAPER_EXT_TRIMP_SCALE,
            "hrrReference": HRR_REFERENCE,
            "decayLambda": 0.30,
            "segmentKm": 0.5,
            "target": "moving_time",
            "status": "optional_not_run_runtime_guard" if not RUN_SEGMENT_LENGTH_SENSITIVITY else "computed",
            "alpha": np.nan,
            "fatigueModel": "",
            "fatigueCoef": np.nan,
            "r2": np.nan,
            "maeMin": np.nan,
            "mapePct": np.nan,
            "biasMin": np.nan,
        }
    )

# Comparator fairness: report available LOO rows on matched held-out activity folds.
matched_rows = []
for source_name, source_df in [
    ("paper_stage", paper_extension_comparison),
    ("linear_regression", extension_comparison),
    ("hr_regression", hr_comparison),
]:
    if source_df.empty:
        continue
    loo_rows = source_df[source_df["stage"].str.contains("LOO", regex=False, na=False)].copy()
    for _, row in loo_rows.iterrows():
        matched_rows.append(
            {
                "checkType": "matched_fold_comparator",
                "validation": "loo",
                "cohort": row["cohort"],
                "model": row["stage"],
                "trimpScale": np.nan,
                "hrrReference": np.nan,
                "decayLambda": np.nan,
                "segmentKm": SEGMENT_KM,
                "target": "moving_time",
                "status": "computed",
                "alpha": np.nan,
                "fatigueModel": source_name,
                "fatigueCoef": np.nan,
                "r2": row["r2"],
                "maeMin": row["maeMin"],
                "mapePct": row["mapePct"],
                "biasMin": row["biasMin"],
            }
        )
robustness_rows.extend(matched_rows)

robustness_checks = pd.DataFrame(robustness_rows)
paper_asset_paths.append(save_table(robustness_checks, "table_robustness_checks.csv"))

# Robustness heatmaps: TRIMP scale by HRR reference, faceted by decay and cohort.
heatmap_df = robustness_checks[
    robustness_checks["checkType"].eq("stage3_grid_sensitivity")
    & robustness_checks["status"].eq("computed")
].copy()
fig, axes = plt.subplots(
    len(cohorts),
    len(ROBUSTNESS_DECAY_LAMBDAS),
    figsize=(4.1 * len(ROBUSTNESS_DECAY_LAMBDAS), 3.5 * len(cohorts)),
    squeeze=False,
)
for row_idx, cohort_name in enumerate(cohorts.keys()):
    for col_idx, decay_lambda in enumerate(ROBUSTNESS_DECAY_LAMBDAS):
        ax = axes[row_idx, col_idx]
        subset = heatmap_df[
            heatmap_df["cohort"].eq(cohort_name) & heatmap_df["decayLambda"].eq(decay_lambda)
        ]
        if subset.empty:
            ax.set_axis_off()
            continue
        pivot = subset.pivot_table(
            index="trimpScale",
            columns="hrrReference",
            values="maeMin",
            aggfunc="min",
        ).reindex(index=ROBUSTNESS_TRIMP_SCALES, columns=ROBUSTNESS_HRR_REFERENCES)
        image = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto", cmap="viridis_r")
        ax.set_xticks(range(len(ROBUSTNESS_HRR_REFERENCES)), [str(v) for v in ROBUSTNESS_HRR_REFERENCES])
        ax.set_yticks(range(len(ROBUSTNESS_TRIMP_SCALES)), [str(v) for v in ROBUSTNESS_TRIMP_SCALES])
        ax.set_xlabel("HRR reference")
        ax.set_ylabel("TRIMP scale")
        ax.set_title(f"{cohort_name}, decay={decay_lambda}")
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                value = pivot.iloc[i, j]
                if pd.notna(value):
                    ax.text(j, i, f"{value:.1f}", ha="center", va="center", color="white", fontsize=8)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="MAE min")
fig.suptitle("Stage 3 robustness: TRIMP scale, HRR reference, and decay", fontsize=14)
fig.tight_layout()
paper_asset_paths.append(save_paper_figure(fig, "fig_robustness_heatmaps.png"))

display(robustness_checks.head(12))


In [ ]:
# Anonymized derived feature exports.
release_ids = sorted(segment_features["activityId"].astype(str).dropna().unique().tolist())
activity_index_map = {activity_id: f"A{idx + 1:04d}" for idx, activity_id in enumerate(release_ids)}

release_activity_df = activity_df[activity_df["activityId"].astype(str).isin(release_ids)].copy()
release_activity_df["activityIndex"] = release_activity_df["activityId"].astype(str).map(activity_index_map)
for cohort_name, cohort_df in cohorts.items():
    cohort_ids = set(cohort_df["activityId"].astype(str))
    release_activity_df[f"cohort_{cohort_name}"] = release_activity_df["activityId"].astype(str).isin(cohort_ids)

activity_export_cols = [
    "activityIndex",
    "cohort_hardTrailRun",
    "cohort_top10HardTrailByHRR",
    "cohort_selectedDateRaces",
    "distanceKm",
    "ascentM",
    "actualTimeSec",
    "elapsedSec",
    "hrReserveRatio",
    "ctl",
    "tsb",
    "trimpRediSlow",
    "trimpRediBalance",
    "ctlReadinessFactor",
    "rediReadinessFactor",
    "meanAltitudeM",
    "technicalityGps",
    "temperatureC",
]
anonymized_activity_features = release_activity_df[
    [col for col in activity_export_cols if col in release_activity_df.columns]
].sort_values("activityIndex")

segment_export_frames = []
if not stage3_segment_predictions.empty:
    source_segments = stage3_segment_predictions.copy()
else:
    source_segments = segment_features.copy()
    source_segments["cohort"] = "allUsableSegments"
    source_segments["stage3PredictedTimeSec"] = np.nan
    source_segments["stage3ResidualSec"] = np.nan
source_segments["activityIndex"] = source_segments["activityId"].astype(str).map(activity_index_map)
segment_export_cols = [
    "cohort",
    "activityIndex",
    "segmentIndex",
    "startKm",
    "endKm",
    "distanceKm",
    "avgGrade",
    "meanAltitudeM",
    "elevGainM",
    "elevLossM",
    "progress",
    "gapFactor",
    "altitudePenalty",
    "meanHrReserve",
    "segmentTrimp",
    "cumTrimpBefore",
    "decayedTrimpBefore",
    "terrainFamily",
    "technicalityCombined",
    "actualTimeSec",
    "stage3PredictedTimeSec",
    "stage3ResidualSec",
    "stage3FatigueState",
    "stage3AcuteTrimpCol",
    "stage3FatigueModel",
]
anonymized_segment_features = source_segments[
    [col for col in segment_export_cols if col in source_segments.columns]
].sort_values(["cohort", "activityIndex", "segmentIndex"])

forbidden_activity_columns = tpm.forbidden_anonymized_columns(anonymized_activity_features.columns)
forbidden_segment_columns = tpm.forbidden_anonymized_columns(anonymized_segment_features.columns)
assert not forbidden_activity_columns, forbidden_activity_columns
assert not forbidden_segment_columns, forbidden_segment_columns

paper_asset_paths.append(save_table(anonymized_activity_features, "anonymized_activity_features.csv"))
paper_asset_paths.append(save_table(anonymized_segment_features, "anonymized_segment_features.csv"))

FEATURE_DESCRIPTIONS = {
    "cohort": "Evaluation cohort label; rows may be repeated across cohorts.",
    "activityIndex": "Anonymous activity index with no link to source platform identifiers.",
    "segmentIndex": "Distance-order segment number inside an anonymized activity.",
    "startKm": "Segment start distance in kilometers.",
    "endKm": "Segment end distance in kilometers.",
    "distanceKm": "Distance in kilometers.",
    "ascentM": "Total positive elevation gain in meters.",
    "actualTimeSec": "Observed moving-time target in seconds.",
    "elapsedSec": "Elapsed-time sensitivity target in seconds when available.",
    "hrReserveRatio": "Activity-level heart-rate reserve ratio.",
    "meanHrReserve": "Segment-level heart-rate reserve ratio.",
    "ctl": "Previous-day TRIMP chronic training load.",
    "tsb": "Previous-day CTL minus ATL balance.",
    "trimpRediSlow": "Slow REDI load state from all activities.",
    "trimpRediBalance": "Slow minus fast REDI balance.",
    "ctlReadinessFactor": "Bounded CTL/TSB readiness multiplier.",
    "rediReadinessFactor": "Bounded REDI readiness multiplier.",
    "meanAltitudeM": "Mean altitude in meters.",
    "technicalityGps": "Activity-level GPS technicality proxy.",
    "temperatureC": "First-party Strava average temperature when available.",
    "avgGrade": "Segment grade ratio.",
    "elevGainM": "Segment elevation gain in meters.",
    "elevLossM": "Segment elevation loss in meters.",
    "progress": "Segment race-progress fraction from 0 to 1.",
    "gapFactor": "Minetti grade-adjustment factor.",
    "altitudePenalty": "One minus the altitude correction factor.",
    "segmentTrimp": "Segment TRIMP computed from segment duration and HRR.",
    "cumTrimpBefore": "Lagged cumulative TRIMP before the segment.",
    "decayedTrimpBefore": "Lagged exponentially decayed TRIMP before the segment.",
    "terrainFamily": "Coarse grade family.",
    "technicalityCombined": "Segment-level technicality feature used by regression diagnostics.",
    "stage3PredictedTimeSec": "Stage 3 segment prediction from the cohort-specific fit.",
    "stage3ResidualSec": "Stage 3 segment prediction minus observed segment moving time.",
    "stage3FatigueState": "Selected Stage 3 acute fatigue state for this cohort.",
    "stage3AcuteTrimpCol": "Source column used as the selected Stage 3 acute fatigue input.",
    "stage3FatigueModel": "Selected Stage 3 fatigue shape for this cohort.",
}
feature_dictionary_rows = []
for dataset_name, df in [
    ("anonymized_activity_features.csv", anonymized_activity_features),
    ("anonymized_segment_features.csv", anonymized_segment_features),
]:
    for col in df.columns:
        feature_dictionary_rows.append(
            {
                "dataset": dataset_name,
                "column": col,
                "description": FEATURE_DESCRIPTIONS.get(col, "Derived manuscript feature."),
            }
        )
feature_dictionary = pd.DataFrame(feature_dictionary_rows)
paper_asset_paths.append(write_markdown_table(feature_dictionary, "table_anonymized_feature_dictionary.md"))

manifest_path = PAPER_ASSET_DIR / "paper_assets_manifest.csv"
manifest_paths = [*paper_asset_paths, manifest_path]
manifest = pd.DataFrame(
    {
        "assetPath": [str(path.relative_to(PROJECT_ROOT)) for path in manifest_paths],
        "exists": [path.exists() if path != manifest_path else True for path in manifest_paths],
    }
).drop_duplicates().sort_values("assetPath")
manifest.to_csv(manifest_path, index=False)

display(manifest)
print(f"Exported {len(manifest)} paper assets to {PAPER_ASSET_DIR}")

